In [4]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
# 오류 발생시
# sklearn.set_config(display="text")

True

In [8]:
base_path = r"/Users/kdt_0900_lyn/ml/resoce/dataset"
food_path = os.path.join(base_path, "brain")

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

os.makedirs(base_path, exist_ok=True)
# https://www.kaggle.com/datasets/trolukovich/food11-image-dataset
api.dataset_download_files(r"masoudnickparvar/brain-tumor-mri-dataset", path=base_path, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset


Task was destroyed but it is pending!
task: <Task pending name='Task-110' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-111' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:564]>
/opt/anaconda3/envs/machine_learning/lib/python3.12/zipfile/__init__.py:516: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def _decodeExtra(self, filename_crc):
Task was destroyed but it is pending!
task: <Task pending name='Task-111' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


다운로드 완료


In [11]:
mean = [0.485, 0.456, 0.406]   # R,G,B 평균 
std = [0.229, 0.224, 0.225]    # R,G,B 표준 편차 

train_transform = transforms.Compose([   # 학습 이미지용 설정 
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, scale=(0.8, 1.2)),  # 원본의 80 ~120% 크기 중 무작위로 선캑 
    transforms.ToTensor(),                         # 텐서 형태로 바꿈 
    transforms.Normalize(mean=mean, std=std)       # 정규화 
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])
 
target_transform = lambda y: torch.tensor([y], dtype=torch.float32)

In [12]:
train_path = os.path.join(output_dir, "brain", "training_set", r"training_set")
test_path = os.path.join(output_dir, "brain", "test_set", r"test_set")

train_dataset = ImageFolder(train_path, transform=train_transform, target_transform = target_transform) 
test_dataset = ImageFolder(test_path, transform=test_transform)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kdt_0900_lyn/ml/resoce/dataset/brain/brain/training_set/training_set'

In [13]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=32)

print(len(train_dataset), len(test_dataset))

NameError: name 'train_dataset' is not defined

In [ ]:
imgs, labels = next(iter(train_dataloader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))

for ax, img, label in zip(axes.flatten(), imgs, labels):
    img = img * std + mean
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(label.item())
    ax.axis("off")

plt.show()